# BSRN Quality Control Demo
# BSRN 质量控制演示

This notebook demonstrates the core functionality of the `bsrn` package, including data reading, clear-sky modeling, and quality control tests.
本笔记本演示了 `bsrn` 包的核心功能，包括数据读取、晴空建模和质量控制测试。

## 1. Import Libraries
## 1. 导入库

In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Add src directory to path so we can import the local bsrn package
# 将 src 目录添加到路径，以便我们可以导入本地 bsrn 包
sys.path.insert(0, os.path.abspath("../src"))
import bsrn

# Set display options
pd.options.display.max_columns = None
%matplotlib inline

## 2. Load Sample Data
## 2. 加载示例数据

In [2]:
# Adjust path to find the data folder from the notebooks directory
stn = "QIQ"
file_path = "../data/QIQ/qiq1224.dat.gz"
if os.path.exists(file_path):
    df = bsrn.io.readers.read_bsrn_station_to_archive(file_path)
    print(f"Data for {stn} loaded successfully (December 2024).")
    display(df.head())
else:
    print(f"File not found: {file_path}. Current working directory: {os.getcwd()}")

## 3. Clear-Sky Modeling
## 3. 晴空建模

In [3]:
# Add clear-sky GHI, BNI, and DHI columns using Ineichen model
if 'df' in locals():
    df = bsrn.physics.clearsky.add_clearsky_columns(df, stn)
    display(df[["ghi", "ghi_clear", "bni", "bni_clear"]].head())

## 4. Quality Control Tests
## 4. 质量控制测试

We apply all standard QC tests using the centralized wrapper.
我们使用集中包装函数运行所有标准 QC 测试。

In [4]:
if 'df' in locals():
    # Run all standard QC tests using the centralized wrapper
    # 使用集中包装函数运行所有标准 QC 测试
    df = bsrn.qc.run_qc(df, station_code=stn)
    
    print("QC tests completed.")
    # Summary of flags (1 indicates failure)
    # 标记摘要（1 表示失败）
    flag_cols = [c for c in df.columns if c.startswith('flag')]
    print("Flag counts (non-zero indicates failure):")
    
    display(df[flag_cols].sum())

## 5. Visualization
## 5. 可视化

In [5]:
if 'df' in locals():
    # Select a day with known QC issues to visualize (December 9, 2024)
    # 选择一个有已知 QC 问题的日期进行可视化 (2024年12月9日)
    day_str = "2024-12-09"
    day_data = df.loc[day_str]
    
    plt.figure(figsize=(14, 7))
    plt.plot(day_data.index, day_data['ghi'], label='Measured GHI', color='#E69F00', linewidth=2)
    plt.plot(day_data.index, day_data['ghi_clear'], label='Clear-Sky GHI', color='#56B4E9', linestyle='--', alpha=0.7)
    
    # Highlight errors (e.g., if tracker was off)
    # 突出显示错误（例如，如果跟踪器失准）
    if 'flagTracker' in day_data.columns:
        tracker_off = day_data[day_data['flagTracker'] > 0]
        if not tracker_off.empty:
            plt.scatter(tracker_off.index, tracker_off['ghi'], color='red', label='Tracker-Off Flag', s=10, zorder=5)
            
    plt.title(f"BSRN Quality Control: {stn} Station ({day_str})", fontsize=16)
    plt.ylabel("Irradiance [W/m^2]", fontsize=12)
    plt.legend(frameon=False)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()